In [ ]:
import datetime as dt
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm import tqdm
from collections import Counter
from imblearn.over_sampling import RandomOverSampler
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler
import sklearn.linear_model
from matplotlib import pyplot
from numpy import where
from sklearn import metrics
from sklearn import preprocessing
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
import sklearn.svm
import sklearn.gaussian_process
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold
from sklearn.model_selection import learning_curve
from sklearn.model_selection import train_test_split
from sklearn.model_selection import validation_curve
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
### LIB TO OPEN EXCEL FILE
import pandas as pd
from matplotlib import pyplot as plt
from pathlib import Path
import joblib

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


## Importing Original Data after removing unwanted columns

In [ ]:
root_dir='/content/gdrive/MyDrive/ResearchProject/'
#data_dir = '/content/gdrive/MyDrive/ResearchProject/data/new_data/'
data_dir="/content/gdrive/MyDrive/ResearchProject/data/labeled_normal_data/new_TLSI_input/all_pids_q=0.2.npy"
output_dir = root_dir + "classification_results_2023/"
#original_data= pd.read_csv(data_dir+"/SB-023_original.csv")


In [ ]:
data_ndarr = np.load(data_dir, allow_pickle=True)
print("data_ndarr type :", type(data_ndarr))

datadict = data_ndarr.item()
print("datadict type :", type(datadict))

data_ndarr type : <class 'numpy.ndarray'>
datadict type : <class 'dict'>


In [ ]:
p_ids = datadict.keys()
smaple_size_dict={}
print("patients: n =", len(p_ids), end='\n\n')

for p_id in p_ids:
    df = datadict[p_id]
    # drop column from input data file
    #df.drop(['id','date','am_pef_org','BMI','sex'], axis=1, inplace=True) #ALL DROPS SHOULD OCCUR HERE
    df.drop(['id','date','am_pef_org','BMI','sex', 'age'], axis=1, inplace=True) #,'id','date','am_pef_org','BMI',  'sex', 'age'
    df = np.asarray(df).astype(np.float32)
    print(p_id, "shape:", df.shape)#, end="\t\t")
    smaple_size_dict[p_id]=df.shape[0]

datadict[p_id].head(4)

patients: n = 26

SB-001 shape: (188, 28)
SB-003 shape: (180, 28)
SB-008 shape: (178, 28)
SB-011 shape: (210, 28)
SB-012 shape: (194, 28)
SB-022 shape: (204, 28)
SB-023 shape: (194, 28)
SB-025 shape: (117, 28)
SB-028 shape: (92, 28)
SB-033 shape: (53, 28)
SB-037 shape: (210, 28)
SB-043 shape: (191, 28)
SB-056 shape: (191, 28)
SB-059 shape: (203, 28)
SB-060 shape: (88, 28)
SB-070 shape: (202, 28)
SB-071 shape: (124, 28)
SB-073 shape: (161, 28)
SB-078 shape: (153, 28)
SB-079 shape: (142, 28)
SB-080 shape: (181, 28)
SB-081 shape: (34, 28)
SB-082 shape: (137, 28)
SB-083 shape: (206, 28)
SB-089 shape: (49, 28)
SB-112 shape: (180, 28)


,income,road_dist,cooking,y_am_pef,tempin,humidin,pm25in,co2in,tempdiffin,humidiffin,...,windsd,humid,varp,dewpt,airp,seap,solrhr,solramnt,grdt,class
1,0.857,0.0,0.167,0.558,0.216,0.513,0.358,0.678,0.562,0.364,...,0.107,0.668,0.174,0.471,0.822,0.819,0.165,0.003,0.206,0.0
2,0.857,0.0,0.167,0.006,0.299,0.521,0.411,0.700,0.436,0.672,...,0.171,0.388,0.190,0.491,0.676,0.670,0.603,0.165,0.383,0.0
3,0.857,0.0,0.167,0.078,0.420,0.713,0.372,0.918,0.469,0.821,...,0.515,0.359,0.197,0.481,0.519,0.518,0.485,0.104,0.346,1.0
4,0.857,0.0,0.167,0.428,0.438,0.468,0.224,0.762,0.560,0.490,...,0.338,0.527,0.175,0.467,0.596,0.596,0.491,0.141,0.339,0.0


In [ ]:
target_grp = ["SB-028", "SB-011", "SB-112", "SB-001", "SB-012", "SB-073", "SB-078", "SB-080", "SB-083", "SB-043", "SB-037", "SB-022", "SB-025", "SB-060", "SB-023", "SB-071", "SB-033", "SB-079", "SB-056", "SB-059", "SB-003", "SB-089", "SB-082", "SB-008", "SB-070"]


## Importing Classification function ( classificationv3.py for splitting the data and also executing classification models)

In [ ]:
import importlib.util
import os
spec = importlib.util.spec_from_file_location('classification', root_dir + 'classification/Base_Model/Classificationv3.py')
module = importlib.util.module_from_spec(spec)

spec.loader.exec_module(module)

In [ ]:
predictColumn = "class"
minorityVariable = 0 # must be an integer


## Method for calling classification algo functions for classifying the data and showing the results

In [ ]:
#EVALUATION_METRICS = ['Weighted Accuracy', 'Sensitivity/Recall', 'Precision_avg', 'F1_avg', 'Specificity', 'Precision', 'Precision_class1', 'F1', 'F1_class1']
EVALUATION_METRICS = ["Weighted Accuracy", "Sensitivity/Recall", "Specificity", "Precision_class0", "Precision_class1", "Precision_avg", "F1_class0", "F1_class1", "F1_avg", "auc_roc_score", "False_Discovery_Rate", "False_Negative_Rate", "False_Omission_Rate", "False_Positive_Rate", "Jaccard"]


In [ ]:
#patient_id="SB-023/"
#file_name="SB-023"
#file_name_synthetic="SB-001_synthetic"

In [ ]:
def classification_func(file_name,target, predictCol, minorVar, category_group, cls, output_dir, df_evaluation_results):
    #runtime()

 #   if len(cls)!=2:
  #    print('process one model each time')
   #   return

    print('~~~~~~~~~~~~~~~~~', file_name,'~~~~~~~~~~~~~~~~~~~~~~')
#+ file_name + '/'
    dataset_dir = Path(output_dir +target+'/'+category_group +  "/diagrams/" ) #RESULT_DIR

    # Create Directory for this patient
    if not (dataset_dir.exists() and dataset_dir.is_dir()):
        os.makedirs(dataset_dir)

    print(dataset_dir)

    save_path = output_dir+ target+'/'+category_group +'/diagrams/' + 'sklearn_models_'
    print(save_path)

    CLASS_VAR = predictCol
    MINORITY_VAR = minorVar
## change the data attribute in the Classify method below for eg: original_data or synthetic_data
    dataset = pd.DataFrame()

    dataset=datadict[target].copy()
    bestDT = module.Classify(dataset,target, True, class_var=CLASS_VAR, minority_var=MINORITY_VAR, drawConfusionMatrix=True, classifiers=cls, scale=True, save_path=save_path, after_split=True)

    confusion_matrixes_file = save_path + "confusion_mat_" + category_group + "_"+ file_name
    module.save_confusion_matrix(bestDT[1], confusion_matrixes_file)
    roc_curve_file = save_path + "roc_curve_" + category_group + "_"+ file_name
    module.save_Draw_Roc(bestDT[2], roc_curve_file)
    if len(bestDT[3])!=0:
      featureImportance_path = save_path + "important_mat_" + category_group + "_"+ file_name
      module.save_Draw_FeatureImportanceMatrix(dataset,target,bestDT[3],cls[0], featureImportance_path, True)

    model_path = save_path + "bestModel_" + category_group + "_"+ file_name + ".joblib"
    joblib.dump(bestDT[4], model_path)
    loaded_model = joblib.load(model_path)
    print(loaded_model)

    df_evaluation_results[file_name] = bestDT[0]
    df_evaluation_results.to_csv(output_dir+target+'/'+category_group+"/sklearn_models_balanced_for_train_metrics_"+category_group + ".csv")


In [ ]:
%run  '/content/gdrive/MyDrive/ResearchProject/classification/Base_Model/Classificationv3.py'

In [ ]:
%run  '/content/gdrive/MyDrive/ResearchProject/BalancingAlgorithms/Balance.py'

## SVM Runner

## SVM

In [ ]:
def SVMRunner(category_group):
  for target in tqdm(p_ids):
    if target in ['SB-081','SB-089','SB-033']:
      continue
    df_evaluation_results = pd.DataFrame(index=EVALUATION_METRICS)
    #category_group = 'SVM'
    file_name=target
    cls = ["SVM", sklearn.svm.SVC(kernel="linear")]
    classification_func(file_name,target, predictColumn, minorityVariable, category_group, cls, output_dir, df_evaluation_results)

In [ ]:
SVMRunner('SVM')

Output hidden; open in https://colab.research.google.com to view.